# 题目四：四杆机构重力势能极小值 —— 分步求解

**题目**：四杆机构，$l_1=l_2=1.0\,\mathrm{m},\ l_3=1.2\,\mathrm{m},\ l_4=2.0\,\mathrm{m}$，
$m_1=m_2=1.0\,\mathrm{kg},\ g=-9.8\,\mathrm{m/s^2}$。
求重力势能取极小值时两质点坐标 $(x_1,y_1)$、$(x_2,y_2)$ 满足的等式方程，并判定解是否为极小值点。

**求解路线（本笔记按此顺序逐步拆开）**：

1. 约束方程 $c(q)=0$（3 个约束：两根杆长 + 末端到固定铰距离）
2. 约束雅可比矩阵 $c_q$（3×4）
3. 重力势能 $f(q)$ 及其梯度 $f_q$
4. KKT 一阶条件 $F(x)=0$：$x=[q;\lambda]\in\mathbb{R}^7$
5. $F(x)$ 的雅可比矩阵 $J(x)$（7×7 分块矩阵）
6. 线性求解器：**列主元高斯消元**（自己实现，不用现成求解器）
7. Newton-Raphson 迭代：每步解 $J\cdot\Delta x=-F$
8. 二阶判据 $Q=d^{T}(H_f+\lambda^{T}H_c)d$（$d$ 为切向方向）
9. 多初值求出全部驻点并判定

> 本笔记与 `lianxi1.cpp`、`lianxi1.m` 功能一致，结果应完全相同。

## 0. 准备：参数设置

In [18]:
import numpy as np

# 四杆机构参数（题目四给定）
l1, l2, l3, l4 = 1.0, 1.0, 1.2, 2.0
m1, m2, g = 1.0, 1.0, -9.8

print(f"l1={l1:.6f}, l2={l2:.6f}, l3={l3:.6f}, l4={l4:.6f}, "
      f"m1={m1:.6f}, m2={m2:.6f}, g={g:.6f}")

l1=1.000000, l2=1.000000, l3=1.200000, l4=2.000000, m1=1.000000, m2=1.000000, g=-9.800000


## 第 1 步：约束方程 $c(q)=0$

广义坐标取两质点的直角坐标 $q=(x_1,\ y_1,\ x_2,\ y_2)$，三个约束分别是：

- 杆 1（原点 → 质点 1）：$x_1^2+y_1^2-l_1^2=0$
- 杆 2（质点 1 → 质点 2）：$(x_2-x_1)^2+(y_2-y_1)^2-l_2^2=0$
- 末端到固定铰 $(l_4,0)$：$(x_2-l_4)^2+y_2^2-l_3^2=0$

In [19]:
# 约束方程 c(q) = 0（3 个约束）
def constraint(q):
    x1, y1, x2, y2 = q
    return np.array([
        x1*x1 + y1*y1 - l1*l1,
        (x2-x1)**2 + (y2-y1)**2 - l2*l2,
        (x2-l4)**2 + y2*y2 - l3*l3,
    ])

q_demo = np.array([0.4, 0.8, 1.2, 0.9])
print("在 q =", q_demo, "处 c(q) =", np.round(constraint(q_demo), 6))

在 q = [0.4 0.8 1.2 0.9] 处 c(q) = [-0.2  -0.35  0.01]


## 第 2 步：约束雅可比矩阵 $c_q=\partial c/\partial q$（3×4）

$$c_q=\begin{bmatrix}
2x_1 & 2y_1 & 0 & 0\\
2(x_1-x_2) & 2(y_1-y_2) & 2(x_2-x_1) & 2(y_2-y_1)\\
0 & 0 & 2(x_2-l_4) & 2y_2
\end{bmatrix}$$

In [20]:
# 约束雅可比矩阵 c_q（3×4）
def constraint_jacobian(q):
    x1, y1, x2, y2 = q
    return np.array([
        [2*x1,        2*y1,        0,             0],
        [2*(x1-x2),   2*(y1-y2),   2*(x2-x1),     2*(y2-y1)],
        [0,           0,           2*(x2-l4),     2*y2],
    ], dtype=float)

print("c_q(q_demo) =\n", constraint_jacobian(q_demo))

c_q(q_demo) =
 [[ 0.8  1.6  0.   0. ]
 [-1.6 -0.2  1.6  0.2]
 [ 0.   0.  -1.6  1.8]]


In [21]:
# 验证：解析雅可比 vs 数值差分（中心差分）
h = 1e-6
C_num = np.zeros((3, 4))
for j in range(4):
    dq = np.zeros(4); dq[j] = h
    C_num[:, j] = (constraint(q_demo + dq) - constraint(q_demo - dq)) / (2*h)

print(f"解析 c_q 与数值差分的最大偏差 = {np.abs(C_num - constraint_jacobian(q_demo)).max():.2e}")

解析 c_q 与数值差分的最大偏差 = 1.76e-10


## 第 3 步：重力势能 $f(q)$ 及其梯度

$$f(q)=m_1 g\,y_1+m_2 g\,y_2,\qquad
f_q=(0,\ m_1 g,\ 0,\ m_2 g)$$

In [22]:
# 重力势能 f(q)
def potential(q):
    x1, y1, x2, y2 = q
    return m1*g*y1 + m2*g*y2

# 势能梯度 f_q（常数向量）
def potential_grad():
    return np.array([0.0, m1*g, 0.0, m2*g])

print("f_q =", potential_grad())

f_q = [ 0.  -9.8  0.  -9.8]


## 第 4 步：KKT 一阶条件

极小值点满足（题目一已推导）：

$$f_q^{T}+c_q^{T}\lambda=0,\qquad c(q)=0$$

合并成对 $x=[q;\ \lambda]\in\mathbb{R}^7$ 的方程：

$$F(x)=\begin{bmatrix} f_q^{T}+c_q^{T}\lambda\\ c(q)\end{bmatrix}=0$$

In [23]:
# 一阶条件（KKT 方程），7 维
def F(x):
    q, lam = x[:4], x[4:]
    return np.concatenate([potential_grad() + constraint_jacobian(q).T @ lam,
                           constraint(q)])

In [24]:
# 验证：把已知解代入，F 应接近 0（数值受 6 位小数舍入限制）
x_known = np.array([0.429506, 0.903064, 1.418683, 1.049795,
                    5.837834, 2.534822, 4.313281])
print(f"已知解处 |F| = {np.linalg.norm(F(x_known)):.2e}  （≈0，说明一阶条件写对了）")

已知解处 |F| = 1.11e-05  （≈0，说明一阶条件写对了）


## 第 5 步：$F(x)$ 的雅可比矩阵 $J(x)$（7×7）

$$J=\begin{bmatrix}
H_f+\sum_i\lambda_i (c_i)_{qq} & c_q^{T}\\
c_q & 0
\end{bmatrix}$$

势能是线性的，所以 $H_f=0$；每个约束的二阶导都是**常数矩阵**：

- $c_1$：$H_{c_1}=\mathrm{diag}(2,2,0,0)$
- $c_2$：对角加 $2$，$(x_1,x_2)$、$(y_1,y_2)$ 交叉项加 $-2$
- $c_3$：$H_{c_3}=\mathrm{diag}(0,0,2,2)$

In [25]:
# H = H_f + Σ λ_i (c_i)_qq（H_f = 0）
def weighted_hessian(lam):
    lam1, lam2, lam3 = lam
    H = np.zeros((4, 4))
    # c1 = x1^2 + y1^2 - l1^2
    H[0, 0] += 2*lam1;  H[1, 1] += 2*lam1
    # c2 = (x2-x1)^2 + (y2-y1)^2 - l2^2
    H[0, 0] += 2*lam2;  H[1, 1] += 2*lam2
    H[2, 2] += 2*lam2;  H[3, 3] += 2*lam2
    H[0, 2] -= 2*lam2;  H[2, 0] -= 2*lam2
    H[1, 3] -= 2*lam2;  H[3, 1] -= 2*lam2
    # c3 = (x2-l4)^2 + y2^2 - l3^2
    H[2, 2] += 2*lam3;  H[3, 3] += 2*lam3
    return H

# F(x) 的雅可比矩阵（7×7）
def J(x):
    q, lam = x[:4], x[4:]
    C = constraint_jacobian(q)
    return np.block([[weighted_hessian(lam), C.T],
                     [C,                  np.zeros((3, 3))]])

In [26]:
# 验证：解析 J vs 数值差分
x_test = np.array([0.5, 0.9, 1.4, 1.0, 2.0, 1.0, 3.0])
J_num = np.zeros((7, 7))
for j in range(7):
    dx = np.zeros(7); dx[j] = h
    J_num[:, j] = (F(x_test + dx) - F(x_test - dx)) / (2*h)

print(f"解析 J 与数值差分的最大偏差 = {np.abs(J_num - J(x_test)).max():.2e}")

解析 J 与数值差分的最大偏差 = 6.58e-10


## 第 6 步：线性求解器 —— 列主元高斯消元

Newton 每步都要解一个线性方程组 $J\cdot\Delta x=-F$，求解器自己实现：

1. 对 $A$ 和 $b$ 做同样的行变换；
2. 对第 $k$ 列，选**绝对值最大**的元素作主元（列主元），必要时交换两行；
3. 用主元行消去下面各行第 $k$ 列的元素（$row_i\leftarrow row_i-\frac{a_{ik}}{a_{kk}}row_k$）；
4. 回代：$x_i=\big(b_i-\sum_{j>i}a_{ij}x_j\big)/a_{ii}$；
5. 主元过小（$\le 10^{-12}$）判定为奇异，返回错误码 3。

In [27]:
# 列主元高斯消元：解 A·x = b。返回 (x, info)，info=0 成功、3 奇异
def guass_solver(A, b, tol=1e-12):
    n = A.shape[0]
    x = np.zeros(n)
    if n == 0:
        return x, 0

    M = A.astype(float).copy()
    rhs = b.astype(float).copy()

    for k in range(n):
        # —— 选主元：第 k 列、第 k 行及以下找绝对值最大者 ——
        p = k + int(np.argmax(np.abs(M[k:, k])))
        if np.abs(M[p, k]) <= tol:
            return x, 3

        # —— 交换主元行（右端项一起换）——
        if p != k:
            M[[k, p]] = M[[p, k]]
            rhs[[k, p]] = rhs[[p, k]]

        pivot = M[k, k]

        # —— 消元 ——
        for i in range(k+1, n):
            factor = M[i, k] / pivot
            if factor == 0.0:
                continue
            M[i, k:] -= factor * M[k, k:]
            rhs[i] -= factor * rhs[k]

    # —— 回代 ——
    for i in range(n-1, -1, -1):
        s = rhs[i] - M[i, i+1:] @ x[i+1:]
        if np.abs(M[i, i]) <= tol:
            return x, 3
        x[i] = s / M[i, i]

    return x, 0

In [28]:
# 测试 1：3×3 方程组（就是之前演示用过的那组）
A_test = np.array([[1., 1., 1.],
                   [2., 1., -1.],
                   [3., 2., 1.]])
b_test = np.array([6., 1., 10.])
x_test_sol, info = guass_solver(A_test, b_test)
print("3×3 测试： x =", x_test_sol, "  info =", info)   # 应为 [1. 2. 3.]

# 测试 2：奇异矩阵（第二行是第一行的 2 倍）
A_sing = np.array([[1., 2.],
                   [2., 4.]])
_, info_sing = guass_solver(A_sing, np.array([1., 2.]))
print("奇异矩阵：info =", info_sing, "（应为 3）")

3×3 测试： x = [1. 2. 3.]   info = 0
奇异矩阵：info = 3 （应为 3）


## 第 7 步：Newton-Raphson 迭代

从初值 $x_0$ 出发反复执行：

$$\text{解 }J(x_k)\,\Delta x=-F(x_k),\qquad x_{k+1}=x_k+\Delta x$$

直到 $\|F\|$ 或步长 $\|\Delta x\|$ 足够小。

> ⚠️ 注意：**乘子初值不能取 0**。当 $\lambda=0$ 时
> $J=\begin{bmatrix}0 & c_q^{T}\\ c_q & 0\end{bmatrix}$ 是奇异的（切向向量 $(d,0)$ 就是它的零空间向量），
> 高斯消元会直接报奇异。所以初值取 $\lambda_0=(1,1,1)$。

In [29]:
# Newton-Raphson：解 J·dx = -F，x ← x + dx
def newton_solve(x, max_iter=100, tol=1e-10, verbose=False):
    x = x.astype(float).copy()
    for it in range(max_iter):
        Fx = F(x)
        nF = np.linalg.norm(Fx)
        if nF < tol:                       # 已满足一阶条件
            return x, True, it

        dx, info = guass_solver(J(x), -Fx)
        if info != 0:                      # 雅可比奇异，无法继续
            return x, False, it

        x = x + dx
        if verbose:
            print(f"  第 {it+1:2d} 次迭代： |F| = {nF:.3e}， |dx| = {np.linalg.norm(dx):.3e}")

        if np.linalg.norm(dx) < 1e-14:     # 步长已到机器精度
            break

    ok = np.linalg.norm(F(x)) < 1e-9       # 最终用残差判定
    return x, ok, it

In [30]:
# 单初值演示：看迭代过程
x0 = np.array([0.5, 0.9, 1.4, 1.0, 1.0, 1.0, 1.0])
print("从初值 (0.5, 0.9, 1.4, 1.0, λ=1,1,1) 出发：")
x_sol, ok, iters = newton_solve(x0, verbose=True)
print(f"\n收敛 = {ok}，共迭代 {iters} 次")
print("解 =", np.round(x_sol, 6))

从初值 (0.5, 0.9, 1.4, 1.0, λ=1,1,1) 出发：
  第  1 次迭代： |F| = 1.123e+01， |dx| = 6.250e+00
  第  2 次迭代： |F| = 1.540e+00， |dx| = 5.251e-01
  第  3 次迭代： |F| = 5.862e-02， |dx| = 2.370e-02
  第  4 次迭代： |F| = 1.030e-04， |dx| = 7.225e-05
  第  5 次迭代： |F| = 4.003e-10， |dx| = 2.773e-10

收敛 = True，共迭代 5 次
解 = [0.429506 0.903064 1.418683 1.049795 5.837834 2.534822 4.313281]


## 第 8 步：二阶判据 $Q$

约束面上任何切向变动 $\delta x=(d,0)$ 满足 $c_q d=0$。题目二的结论：

$$Q=d^{T}\big(H_f+\lambda^{T}H_c\big)d
\ \begin{cases}>0 & \text{严格局部极小}\\<0 & \text{不是极小（鞍点）}\end{cases}$$

3×4 的 $c_q$ 切空间是 1 维，方向 $d$ 用"广义叉积"构造：
$d_i=(-1)^{i}\det(\text{去掉第 } i \text{ 列的 } 3\times3 \text{ 子式})$（0 基下标）。

In [31]:
# 切向量 d：3×4 矩阵的广义叉积
def tangent_direction(C):
    d = np.zeros(4)
    for i in range(4):
        cols = [j for j in range(4) if j != i]
        minor = C[:, cols]
        d[i] = (1.0 if i % 2 == 0 else -1.0) * np.linalg.det(minor)
    return d

# 二阶判据 Q = dᵀ(H_f + ΣλᵢH_ci)d
def second_order_q(x):
    q, lam = x[:4], x[4:]
    d = tangent_direction(constraint_jacobian(q))
    return d @ weighted_hessian(lam) @ d

In [32]:
# 用已知解演示：切向量确实与 c_q 的各行正交
C_known = constraint_jacobian(x_known[:4])
d_known = tangent_direction(C_known)
print("切向量 d =", np.round(d_known, 4))
print("c_q @ d =", np.round(C_known @ d_known, 6), " （≈0，d 是切向方向）")

切向量 d = [ 8.1184 -3.8612  6.9729  3.8612]
c_q @ d = [ 0.  0. -0.]  （≈0，d 是切向方向）


## 第 9 步：多初值求全部驻点

一阶条件是非线性方程组，可能有多个解；从 4 个初值分别出发求解，收集互不重复的驻点。

In [33]:
starts = np.array([
    [0.5,  0.9, 1.4,  1.0],
    [0.5, -0.9, 1.4, -1.0],
    [0.3,  1.0, 1.1,  1.1],
    [0.3, -1.0, 1.1, -1.1],
])

sols = []
for s, q0 in enumerate(starts, 1):
    x0 = np.concatenate([q0, [1.0, 1.0, 1.0]])
    x_sol, ok, _ = newton_solve(x0)
    if not ok:
        print(f"初值 {s}: {q0}  ->  未收敛，跳过")
        continue

    print(f"初值 {s}: {q0}  ->  收敛到 ({x_sol[0]:.6f}, {x_sol[1]:.6f}, "
          f"{x_sol[2]:.6f}, {x_sol[3]:.6f})")

    # 与已有解去重
    if all(np.linalg.norm(x_sol - xs) > 1e-6 for xs in sols):
        sols.append(x_sol)

print(f"\n共找到 {len(sols)} 个互不重复的驻点解")

初值 1: [0.5 0.9 1.4 1. ]  ->  收敛到 (0.429506, 0.903064, 1.418683, 1.049795)
初值 2: [ 0.5 -0.9  1.4 -1. ]  ->  收敛到 (0.429506, -0.903064, 1.418683, -1.049795)
初值 3: [0.3 1.  1.1 1.1]  ->  收敛到 (0.429506, 0.903064, 1.418683, 1.049795)
初值 4: [ 0.3 -1.   1.1 -1.1]  ->  收敛到 (0.429506, -0.903064, 1.418683, -1.049795)

共找到 2 个互不重复的驻点解


## 最终结果：驻点 + 判据判定

In [34]:
print("========== 题目四：四杆机构势能极小值 ==========")
for k, x in enumerate(sols, 1):
    q, lam = x[:4], x[4:]
    f = potential(q)
    Q = second_order_q(x)

    verdict = ("极小值点（稳定）"   if Q > 1e-8 else
               "非极小值点（不稳定/鞍点）" if Q < -1e-8 else
               "退化，需看更高阶")

    print(f"\n解 {k}：")
    print(f"  (x1, y1) = ({q[0]:.6f}, {q[1]:.6f})")
    print(f"  (x2, y2) = ({q[2]:.6f}, {q[3]:.6f})")
    print(f"  (λ1, λ2, λ3) = ({lam[0]:.6f}, {lam[1]:.6f}, {lam[2]:.6f})")
    print(f"  势能 f = {f:.6f}")
    print(f"  二阶判据 Q = {Q:.6f}  ->  {verdict}")

========== 题目四：四杆机构势能极小值 ==========

解 1：
  (x1, y1) = (0.429506, 0.903064)
  (x2, y2) = (1.418683, 1.049795)
  (λ1, λ2, λ3) = (5.837834, 2.534822, 4.313281)
  势能 f = -19.138020
  二阶判据 Q = 1800.622182  ->  极小值点（稳定）

解 2：
  (x1, y1) = (0.429506, -0.903064)
  (x2, y2) = (1.418683, -1.049795)
  (λ1, λ2, λ3) = (-5.837834, -2.534822, -4.313281)
  势能 f = 19.138020
  二阶判据 Q = -1800.622182  ->  非极小值点（不稳定/鞍点）


## 结果小结

- 机构一共有两个驻点（关于 $x$ 轴对称）：
  - **$y>0$ 的解**：$f=-19.138020$，$Q=+1800.622182>0$ → **严格局部极小值点**；
  - **$y<0$ 的解**：$f=+19.138020$，$Q=-1800.622182<0$ → 不是极小值点（鞍点）。
- 物理意义：重力势能在较低的那组构型（$f$ 最小）处取极小，另一组是"倒过来的"不稳定平衡位形。
- 与 C++ 版（`lianxi1.cpp`）、MATLAB 版（`lianxi1.m`）的数值逐位一致，互为验证。